In [26]:
import pickle 
import mlflow 
import mlflow.sklearn


import pandas as pd 

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


from sklearn.pipeline import make_pipeline

In [27]:
import mlflow 

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("green-taxi-duration")

<Experiment: artifact_location='s3:/demo-mlflow-s3/1', creation_time=1759607293226, experiment_id='1', last_update_time=1759607293226, lifecycle_stage='active', name='green-taxi-duration', tags={}>

In [28]:
def read_dataframe(filename:str):
    df=pd.read_parquet(filename)
    
    df['duration']=df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    categorical=['PULocationID', 'DOLocationID']
    df[categorical]=df[categorical].astype(str)
    return df


In [29]:
def prepare_dict(df: pd.DataFrame):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical=["PU_DO"]
    numerical=["trip_distance"]
    dicts=df[categorical + numerical].to_dict(orient='records')
    return dicts

In [30]:
df_train=read_dataframe("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet")
df_val=read_dataframe("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet")

target="duration"
y_train=df_train[target].values
y_val=df_val[target].values

dict_train=prepare_dict(df_train)
dict_val=prepare_dict(df_val)

In [32]:
from mlflow.models.signature import infer_signature

signature = infer_signature(dict_train, pipeline.predict(dict_train))

In [34]:
with mlflow.start_run():
    params=dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=13)
    mlflow.log_params(params)
    
    pipeline=make_pipeline(
        DictVectorizer(),
        RandomForestRegressor(**params, n_jobs=-1)
    )
    
    pipeline.fit(dict_train, y_train)
    y_pred= pipeline.predict(dict_val)
    
    rmse=mean_squared_error(y_pred, y_val, squared=False)
    print(params, rmse)
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(
    sk_model=pipeline,
    name="models",
    signature=signature,
    input_example=dict_train.loc[:5]
    )
    

{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 13} 6.756305038881697
🏃 View run peaceful-hound-575 at: http://127.0.0.1:5000/#/experiments/1/runs/a792db4e7e4547ce808a01abdc953227
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


AttributeError: 'list' object has no attribute 'loc'